In [1]:
import sqlite3
import pandas as pd
import numpy as np
import os

In [4]:
# Define the database filename
db_filename = 'loans.db'

# Connect to the SQLite database
conn = sqlite3.connect(db_filename)

# Define the chunk size (adjust based on your system's memory)
chunk_size = 10000

# Create a SQL query string for accepted loans
sql_query = "SELECT * FROM accepted_loans"

# Iterate over the accepted loans table in chunks
chunk_iter = pd.read_sql_query(sql_query, conn, chunksize=chunk_size)

for i, chunk in enumerate(chunk_iter):
    print(f"Processing chunk {i+1}")
    
    # -- Cleaning Step 1: Convert Date Columns --
    date_cols = ['issue_d', 'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d']
    for col in date_cols:
        if col in chunk.columns:
            chunk[col] = pd.to_datetime(chunk[col], errors='coerce')
    
    # -- Cleaning Step 2: Clean the 'term' Column --
    if 'term' in chunk.columns:
        chunk['term_numeric'] = chunk['term'].str.extract('(\d+)').astype(float)
    
    # -- Cleaning Step 3: Create a Quarterly Indicator Column --
    if 'issue_d' in chunk.columns:
        chunk['issue_quarter'] = chunk['issue_d'].dt.to_period('Q').astype(str)
    
    # Write the cleaned chunk into a new table in the database
    if i == 0:
        # Create new table (replace if exists)
        chunk.to_sql("cleaned_accepted_loans", conn, if_exists="replace", index=False)
    else:
        # Append to the existing table
        chunk.to_sql("cleaned_accepted_loans", conn, if_exists="append", index=False)
    
    print(f"Chunk {i+1} processed and saved into the database.")

# Close the connection when done
conn.close()

print("All chunks processed and saved to the new table 'cleaned_accepted_loans' in the database.")



Processing chunk 1
Chunk 1 processed and saved into the database.
Processing chunk 2
Chunk 2 processed and saved into the database.
Processing chunk 3
Chunk 3 processed and saved into the database.
Processing chunk 4
Chunk 4 processed and saved into the database.
Processing chunk 5
Chunk 5 processed and saved into the database.
Processing chunk 6
Chunk 6 processed and saved into the database.
Processing chunk 7
Chunk 7 processed and saved into the database.
Processing chunk 8
Chunk 8 processed and saved into the database.
Processing chunk 9
Chunk 9 processed and saved into the database.
Processing chunk 10
Chunk 10 processed and saved into the database.
Processing chunk 11
Chunk 11 processed and saved into the database.
Processing chunk 12
Chunk 12 processed and saved into the database.
Processing chunk 13
Chunk 13 processed and saved into the database.
Processing chunk 14
Chunk 14 processed and saved into the database.
Processing chunk 15
Chunk 15 processed and saved into the database

In [ ]:
# Define the database filename
db_filename = 'loans.db'
conn = sqlite3.connect(db_filename)

# ========= Accepted Loans Summary =========

# If the accepted loans table is huge, use chunk iteration; otherwise, you can load it entirely.
# Here, we show a chunk-based approach.
accepted_sql = "SELECT * FROM cleaned_accepted_loans"
chunk_size = 10000

# Initialize a list to store descriptive statistics from each chunk.
accepted_summary_list = []

# Process the accepted loans in chunks
for i, chunk in enumerate(pd.read_sql_query(accepted_sql, conn, chunksize=chunk_size)):
    print(f"Processing accepted loans chunk {i+1}")
    # Compute summary statistics for numeric columns
    chunk_desc = chunk.describe(include='all')
    # Append the chunk's summary to our list
    accepted_summary_list.append(chunk_desc)

# Combine the summaries.
# Note: For a large dataset, you might want to compute running aggregates.
# Here, we simply concatenate the chunk summaries for review.
accepted_summary = pd.concat(accepted_summary_list, axis=1)
accepted_summary.to_csv("accepted_loans_summary.csv")
print("Accepted loans summary statistics saved to 'accepted_loans_summary.csv'.")

# ========= Rejected Loans Summary =========

# Assuming the rejected loans dataset is smaller, load it in one go
rejected_sql = "SELECT * FROM rejected_loans"
rejected_df = pd.read_sql_query(rejected_sql, conn)

# Compute descriptive statistics
rejected_summary = rejected_df.describe(include='all')
rejected_summary.to_csv("rejected_loans_summary.csv")
print("Rejected loans summary statistics saved to 'rejected_loans_summary.csv'.")

# ========= Additional: Summary for Categorical Variables =========
# For accepted loans, it might be useful to look at the frequency counts of key categorical columns.
accepted_df = pd.read_sql_query("SELECT * FROM cleaned_accepted_loans LIMIT 100000", conn)
categorical_cols = ['term', 'grade', 'sub_grade', 'home_ownership', 'verification_status', 'loan_status', 'purpose', 'addr_state']

for col in categorical_cols:
    if col in accepted_df.columns:
        freq = accepted_df[col].value_counts(dropna=False)
        freq.to_csv(f"accepted_{col}_frequency.csv")
        print(f"Frequency counts for {col} saved to 'accepted_{col}_frequency.csv'.")

# Similarly for rejected loans categorical columns
rejected_categorical_cols = ['Loan Title', 'State', 'Policy Code']
for col in rejected_categorical_cols:
    if col in rejected_df.columns:
        freq = rejected_df[col].value_counts(dropna=False)
        freq.to_csv(f"rejected_{col}_frequency.csv")
        print(f"Frequency counts for {col} saved to 'rejected_{col}_frequency.csv'.")

conn.close()

Processing accepted loans chunk 1
Processing accepted loans chunk 2
Processing accepted loans chunk 3
Processing accepted loans chunk 4
Processing accepted loans chunk 5
Processing accepted loans chunk 6
Processing accepted loans chunk 7
Processing accepted loans chunk 8
Processing accepted loans chunk 9
Processing accepted loans chunk 10
Processing accepted loans chunk 11
Processing accepted loans chunk 12
Processing accepted loans chunk 13
Processing accepted loans chunk 14
Processing accepted loans chunk 15
Processing accepted loans chunk 16
Processing accepted loans chunk 17
Processing accepted loans chunk 18
Processing accepted loans chunk 19
Processing accepted loans chunk 20
Processing accepted loans chunk 21
Processing accepted loans chunk 22
Processing accepted loans chunk 23
Processing accepted loans chunk 24
Processing accepted loans chunk 25
Processing accepted loans chunk 26
Processing accepted loans chunk 27
Processing accepted loans chunk 28
Processing accepted loans chu

In [3]:
# Define the database filename
db_filename = 'loans.db'

# Connect to the SQLite database
conn = sqlite3.connect(db_filename)

# Define the chunk size (adjust based on your system's memory)
chunk_size = 10000

# SQL query for the cleaned accepted loans table we created earlier
sql_query_cleaned = "SELECT * FROM cleaned_accepted_loans"

# Prepare an empty list to accumulate filtered chunks
bnpl_chunks = []

# Iterate over the cleaned accepted loans table in chunks
chunk_iter = pd.read_sql_query(sql_query_cleaned, conn, chunksize=chunk_size)

for i, chunk in enumerate(chunk_iter):
    print(f"Filtering chunk {i+1}")

    # -- Additional Filtering: Adjusted BNPL-like Product Characteristics --
    # Filter by loan amount between $1,000 and $8,000 (bottom 25% of the data)
    chunk = chunk[(chunk['loan_amnt'] >= 1000) & (chunk['loan_amnt'] <= 8000)]
    
    # Filter by term: assume BNPL-like loans should have a term of <= 36 months
    if 'term_numeric' in chunk.columns:
        chunk = chunk[chunk['term_numeric'] <= 36]
    
    # Optional: Add a filter on 'purpose' if needed (e.g., consumer retail-related loans)
    # purpose_whitelist = ['credit_card', 'debt_consolidation', 'major_purchase', 'other']
    # chunk = chunk[chunk['purpose'].isin(purpose_whitelist)]
    
    # If the chunk is not empty after filtering, append it
    if not chunk.empty:
        bnpl_chunks.append(chunk)
        print(f"Chunk {i+1}: {len(chunk)} rows retained after adjusted BNPL-like filtering.")
    else:
        print(f"Chunk {i+1}: No rows passed the adjusted BNPL-like filters.")

# Concatenate all filtered chunks into one DataFrame
if bnpl_chunks:
    bnpl_like_df = pd.concat(bnpl_chunks, ignore_index=True)
else:
    bnpl_like_df = pd.DataFrame()

# Write the BNPL-like accepted loans to a new table in the database
bnpl_like_df.to_sql("bnpl_like_accepted_loans", conn, if_exists="replace", index=False)

print("Filtered BNPL-like accepted loans saved into the database as 'bnpl_like_accepted_loans'.")

# Close the connection
conn.close()




Filtering chunk 1
Chunk 1: 2561 rows retained after adjusted BNPL-like filtering.
Filtering chunk 2
Chunk 2: 2612 rows retained after adjusted BNPL-like filtering.
Filtering chunk 3
Chunk 3: 2600 rows retained after adjusted BNPL-like filtering.
Filtering chunk 4
Chunk 4: 2431 rows retained after adjusted BNPL-like filtering.
Filtering chunk 5
Chunk 5: 2509 rows retained after adjusted BNPL-like filtering.
Filtering chunk 6
Chunk 6: 2506 rows retained after adjusted BNPL-like filtering.
Filtering chunk 7
Chunk 7: 2419 rows retained after adjusted BNPL-like filtering.
Filtering chunk 8
Chunk 8: 2464 rows retained after adjusted BNPL-like filtering.
Filtering chunk 9
Chunk 9: 2594 rows retained after adjusted BNPL-like filtering.
Filtering chunk 10
Chunk 10: 2434 rows retained after adjusted BNPL-like filtering.
Filtering chunk 11
Chunk 11: 2300 rows retained after adjusted BNPL-like filtering.
Filtering chunk 12
Chunk 12: 2533 rows retained after adjusted BNPL-like filtering.
Filtering 

/var/folders/bm/0wq19v017y79n8fk7_72tt4c0000gn/T/ipykernel_56445/1738208915.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  bnpl_like_df = pd.concat(bnpl_chunks, ignore_index=True)


Filtered BNPL-like accepted loans saved into the database as 'bnpl_like_accepted_loans'.


In [3]:
# Define the database filename
db_filename = 'loans.db'

# Connect to the SQLite database
conn = sqlite3.connect(db_filename)

# Define the chunk size (adjust based on your system's memory)
chunk_size = 10000

# SQL query for the cleaned accepted loans table we created earlier
sql_query_cleaned = "SELECT * FROM cleaned_accepted_loans"

# Prepare an empty list to accumulate filtered chunks
bnpl_chunks = []

# Iterate over the cleaned accepted loans table in chunks
chunk_iter = pd.read_sql_query(sql_query_cleaned, conn, chunksize=chunk_size)

for i, chunk in enumerate(chunk_iter):
    print(f"Filtering chunk {i+1}")

    # -- Additional Filtering: Adjusted BNPL-like Product Characteristics --
    # Filter by loan amount between $1,000 and $8,000 (bottom 25% of the data)
    chunk = chunk[(chunk['loan_amnt'] >= 1000) & (chunk['loan_amnt'] <= 5000)]
    chunk = chunk[chunk['loan_status']!= "Current"]
    
    # Filter by term: assume BNPL-like loans should have a term of <= 36 months
    if 'term_numeric' in chunk.columns:
        chunk = chunk[chunk['term_numeric'] <= 36]
    
    # Optional: Add a filter on 'purpose' if needed (e.g., consumer retail-related loans)
    # purpose_whitelist = ['credit_card', 'debt_consolidation', 'major_purchase', 'other']
    # chunk = chunk[chunk['purpose'].isin(purpose_whitelist)]
    
    # If the chunk is not empty after filtering, append it
    if not chunk.empty:
        bnpl_chunks.append(chunk)
        print(f"Chunk {i+1}: {len(chunk)} rows retained after adjusted BNPL-like filtering.")
    else:
        print(f"Chunk {i+1}: No rows passed the adjusted BNPL-like filters.")

# Concatenate all filtered chunks into one DataFrame
if bnpl_chunks:
    bnpl_like_df = pd.concat(bnpl_chunks, ignore_index=True)
else:
    bnpl_like_df = pd.DataFrame()

# Write the BNPL-like accepted loans to a new table in the database
bnpl_like_df.to_sql("bnpl_like_accepted_loans_limited", conn, if_exists="replace", index=False)

print("Filtered BNPL-like accepted loans saved into the database as 'bnpl_like_accepted_loans_limited'.")

# Close the connection
conn.close()

Filtering chunk 1
Chunk 1: 1247 rows retained after adjusted BNPL-like filtering.
Filtering chunk 2
Chunk 2: 1292 rows retained after adjusted BNPL-like filtering.
Filtering chunk 3
Chunk 3: 1308 rows retained after adjusted BNPL-like filtering.
Filtering chunk 4
Chunk 4: 1212 rows retained after adjusted BNPL-like filtering.
Filtering chunk 5
Chunk 5: 1262 rows retained after adjusted BNPL-like filtering.
Filtering chunk 6
Chunk 6: 1248 rows retained after adjusted BNPL-like filtering.
Filtering chunk 7
Chunk 7: 1157 rows retained after adjusted BNPL-like filtering.
Filtering chunk 8
Chunk 8: 1126 rows retained after adjusted BNPL-like filtering.
Filtering chunk 9
Chunk 9: 1259 rows retained after adjusted BNPL-like filtering.
Filtering chunk 10
Chunk 10: 1099 rows retained after adjusted BNPL-like filtering.
Filtering chunk 11
Chunk 11: 1076 rows retained after adjusted BNPL-like filtering.
Filtering chunk 12
Chunk 12: 1146 rows retained after adjusted BNPL-like filtering.
Filtering 

/var/folders/bm/0wq19v017y79n8fk7_72tt4c0000gn/T/ipykernel_7055/3001678939.py:44: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  bnpl_like_df = pd.concat(bnpl_chunks, ignore_index=True)


Filtered BNPL-like accepted loans saved into the database as 'bnpl_like_accepted_loans_limited'.


In [ ]:
db_filename = 'loans.db'
conn = sqlite3.connect(db_filename)

# ========= Accepted Loans Summary =========

# If the accepted loans table is huge, use chunk iteration; otherwise, you can load it entirely.
# Here, we show a chunk-based approach.
accepted_sql = "SELECT * FROM bnpl_like_accepted_loans_limited"
chunk_size = 10000

# Initialize a list to store descriptive statistics from each chunk.
accepted_summary_list = []

# Process the accepted loans in chunks
for i, chunk in enumerate(pd.read_sql_query(accepted_sql, conn, chunksize=chunk_size)):
    print(f"Processing accepted loans chunk {i+1}")
    # Compute summary statistics for numeric columns
    chunk_desc = chunk.describe(include='all')
    # Append the chunk's summary to our list
    accepted_summary_list.append(chunk_desc)

# Combine the summaries.
# Note: For a large dataset, you might want to compute running aggregates.
# Here, we simply concatenate the chunk summaries for review.
accepted_summary = pd.concat(accepted_summary_list, axis=1)
accepted_summary.to_csv("bnpl_like_accepted_loans_limited_summary.csv")
print("Accepted loans summary statistics saved to 'bnpl_like_accepted_loans_limited_summary.csv'.")



In [ ]:
# ========= Additional: Summary for Categorical Variables =========
# For accepted loans, it might be useful to look at the frequency counts of key categorical columns.
accepted_df = pd.read_sql_query("SELECT * FROM cleaned_accepted_loans LIMIT 100000", conn)
categorical_cols = ['term', 'grade', 'sub_grade', 'home_ownership', 'verification_status', 'loan_status', 'purpose', 'addr_state']

for col in categorical_cols:
    if col in accepted_df.columns:
        freq = accepted_df[col].value_counts(dropna=False)
        freq.to_csv(f"accepted_{col}_frequency.csv")
        print(f"Frequency counts for {col} saved to 'accepted_{col}_frequency.csv'.")

# Similarly for rejected loans categorical columns
rejected_categorical_cols = ['Loan Title', 'State', 'Policy Code']
for col in rejected_categorical_cols:
    if col in rejected_df.columns:
        freq = rejected_df[col].value_counts(dropna=False)
        freq.to_csv(f"rejected_{col}_frequency.csv")
        print(f"Frequency counts for {col} saved to 'rejected_{col}_frequency.csv'.")

conn.close()

In [4]:
db_filename = 'loans.db'
conn = sqlite3.connect(db_filename)

accepted_sql = "SELECT * FROM bnpl_like_accepted_loans_limited"
chunk_size = 10000

# Dictionary to hold running aggregates for each numeric column.
numeric_summary = {}
first_chunk = True

# Process the data in chunks.
for i, chunk in enumerate(pd.read_sql_query(accepted_sql, conn, chunksize=chunk_size)):
    print(f"Processing accepted loans chunk {i+1}")
    
    # Get the list of numeric columns in this chunk.
    numeric_cols = chunk.select_dtypes(include=[np.number]).columns.tolist()
    
    for col in numeric_cols:
        # Drop missing values for this column.
        col_data = chunk[col].dropna()
        
        if col_data.empty:
            continue  # Skip if no valid data in this chunk for the column.
        
        # For the first chunk, initialize the aggregator.
        if first_chunk or col not in numeric_summary:
            numeric_summary[col] = {
                'count': col_data.count(),
                'sum': col_data.sum(),
                'sum_sq': (col_data ** 2).sum(),
                'min': col_data.min(),
                'max': col_data.max()
            }
        else:
            numeric_summary[col]['count'] += col_data.count()
            numeric_summary[col]['sum'] += col_data.sum()
            numeric_summary[col]['sum_sq'] += (col_data ** 2).sum()
            numeric_summary[col]['min'] = min(numeric_summary[col]['min'], col_data.min())
            numeric_summary[col]['max'] = max(numeric_summary[col]['max'], col_data.max())
    
    first_chunk = False

# After processing all chunks, compute overall summary statistics.
overall_stats = {}
for col, stats in numeric_summary.items():
    count = stats['count']
    if count > 0:
        mean = stats['sum'] / count
        # Variance formula: Var = (sum_sq - (sum^2)/count) / (count - 1)
        variance = (stats['sum_sq'] - (stats['sum'] ** 2) / count) / (count - 1) if count > 1 else np.nan
        std = np.sqrt(variance) if variance >= 0 else np.nan
    else:
        mean = np.nan
        std = np.nan
    overall_stats[col] = {
        'count': count,
        'mean': mean,
        'std': std,
        'min': stats['min'],
        'max': stats['max']
    }

# Convert the overall statistics to a DataFrame.
overall_stats_df = pd.DataFrame(overall_stats).transpose()

# Save the summary statistics to CSV.
overall_stats_df.to_csv("bnpl_like_accepted_loans_limited_numeric_summary.csv")

print("Overall numeric summary statistics saved to 'bnpl_like_accepted_loans_limited_numeric_summary.csv'.")
conn.close()


Processing accepted loans chunk 1
Processing accepted loans chunk 2
Processing accepted loans chunk 3
Processing accepted loans chunk 4
Processing accepted loans chunk 5
Processing accepted loans chunk 6
Processing accepted loans chunk 7
Processing accepted loans chunk 8
Processing accepted loans chunk 9
Processing accepted loans chunk 10
Processing accepted loans chunk 11
Processing accepted loans chunk 12
Processing accepted loans chunk 13
Processing accepted loans chunk 14
Processing accepted loans chunk 15
Processing accepted loans chunk 16
Processing accepted loans chunk 17
Processing accepted loans chunk 18
Processing accepted loans chunk 19
Overall numeric summary statistics saved to 'bnpl_like_accepted_loans_limited_numeric_summary.csv'.


In [5]:
db_filename = 'loans.db'
conn = sqlite3.connect(db_filename)

accepted_sql = "SELECT * FROM bnpl_like_accepted_loans_limited"
chunk_size = 10000

# Dictionary to accumulate frequency counts for each categorical column.
categorical_counts = {}

# Process the data in chunks.
for i, chunk in enumerate(pd.read_sql_query(accepted_sql, conn, chunksize=chunk_size)):
    print(f"Processing categorical chunk {i+1}")
    
    # Identify categorical columns.
    # You might need to adjust the selection depending on how your columns are stored.
    categorical_cols = chunk.select_dtypes(include=['object', 'category']).columns.tolist()
    
    for col in categorical_cols:
        # Get frequency counts for this chunk (including NaNs if desired)
        counts = chunk[col].value_counts(dropna=False)
        
        # Update the running counts.
        for category, count in counts.items():
            # Initialize the column in the dictionary if needed.
            if col not in categorical_counts:
                categorical_counts[col] = {}
            # Update count for the category.
            categorical_counts[col][category] = categorical_counts[col].get(category, 0) + count

conn.close()

# Now, compile a summary DataFrame for categorical columns.
summary_list = []
for col, counts_dict in categorical_counts.items():
    # Convert the counts dictionary to a Series for easier analysis.
    freq_series = pd.Series(counts_dict)
    
    total_count = freq_series.sum()
    unique_count = freq_series.count()
    mode_value = freq_series.idxmax()  # Category with the highest frequency
    mode_count = freq_series.max()
    
    # Create a DataFrame row for this column.
    summary_list.append({
        'column': col,
        'total_count': total_count,
        'unique_categories': unique_count,
        'mode': mode_value,
        'mode_count': mode_count
    })

categorical_summary_df = pd.DataFrame(summary_list)
categorical_summary_df.to_csv("bnpl_like_accepted_loans_limited_categorical_summary.csv", index=False)

print("Overall categorical summary statistics saved to 'bnpl_like_accepted_loans_limited_categorical_summary.csv'.")


Processing categorical chunk 1
Processing categorical chunk 2
Processing categorical chunk 3
Processing categorical chunk 4
Processing categorical chunk 5
Processing categorical chunk 6
Processing categorical chunk 7
Processing categorical chunk 8
Processing categorical chunk 9
Processing categorical chunk 10
Processing categorical chunk 11
Processing categorical chunk 12
Processing categorical chunk 13
Processing categorical chunk 14
Processing categorical chunk 15
Processing categorical chunk 16
Processing categorical chunk 17
Processing categorical chunk 18
Processing categorical chunk 19
Overall categorical summary statistics saved to 'bnpl_like_accepted_loans_limited_categorical_summary.csv'.


In [7]:
db_filename = 'loans.db'
conn = sqlite3.connect(db_filename)

accepted_sql = "SELECT * FROM bnpl_like_accepted_loans"
chunk_size = 10000

# Dictionary to hold running aggregates for each numeric column.
numeric_summary = {}
first_chunk = True

# Process the data in chunks.
for i, chunk in enumerate(pd.read_sql_query(accepted_sql, conn, chunksize=chunk_size)):
    print(f"Processing accepted loans chunk {i+1}")
    
    # Get the list of numeric columns in this chunk.
    numeric_cols = chunk.select_dtypes(include=[np.number]).columns.tolist()
    
    for col in numeric_cols:
        # Drop missing values for this column.
        col_data = chunk[col].dropna()
        
        if col_data.empty:
            continue  # Skip if no valid data in this chunk for the column.
        
        # For the first chunk, initialize the aggregator.
        if first_chunk or col not in numeric_summary:
            numeric_summary[col] = {
                'count': col_data.count(),
                'sum': col_data.sum(),
                'sum_sq': (col_data ** 2).sum(),
                'min': col_data.min(),
                'max': col_data.max()
            }
        else:
            numeric_summary[col]['count'] += col_data.count()
            numeric_summary[col]['sum'] += col_data.sum()
            numeric_summary[col]['sum_sq'] += (col_data ** 2).sum()
            numeric_summary[col]['min'] = min(numeric_summary[col]['min'], col_data.min())
            numeric_summary[col]['max'] = max(numeric_summary[col]['max'], col_data.max())
    
    first_chunk = False

# After processing all chunks, compute overall summary statistics.
overall_stats = {}
for col, stats in numeric_summary.items():
    count = stats['count']
    if count > 0:
        mean = stats['sum'] / count
        # Variance formula: Var = (sum_sq - (sum^2)/count) / (count - 1)
        variance = (stats['sum_sq'] - (stats['sum'] ** 2) / count) / (count - 1) if count > 1 else np.nan
        std = np.sqrt(variance) if variance >= 0 else np.nan
    else:
        mean = np.nan
        std = np.nan
    overall_stats[col] = {
        'count': count,
        'mean': mean,
        'std': std,
        'min': stats['min'],
        'max': stats['max']
    }

# Convert the overall statistics to a DataFrame.
overall_stats_df = pd.DataFrame(overall_stats).transpose()

# Save the summary statistics to CSV.
overall_stats_df.to_csv("bnpl_like_accepted_loans_numeric_summary.csv")

print("Overall numeric summary statistics saved to 'bnpl_like_accepted_loans_numeric_summary.csv'.")
conn.close()

Processing accepted loans chunk 1
Processing accepted loans chunk 2
Processing accepted loans chunk 3
Processing accepted loans chunk 4
Processing accepted loans chunk 5
Processing accepted loans chunk 6
Processing accepted loans chunk 7
Processing accepted loans chunk 8
Processing accepted loans chunk 9
Processing accepted loans chunk 10
Processing accepted loans chunk 11
Processing accepted loans chunk 12
Processing accepted loans chunk 13
Processing accepted loans chunk 14
Processing accepted loans chunk 15
Processing accepted loans chunk 16
Processing accepted loans chunk 17
Processing accepted loans chunk 18
Processing accepted loans chunk 19
Processing accepted loans chunk 20
Processing accepted loans chunk 21
Processing accepted loans chunk 22
Processing accepted loans chunk 23
Processing accepted loans chunk 24
Processing accepted loans chunk 25
Processing accepted loans chunk 26
Processing accepted loans chunk 27
Processing accepted loans chunk 28
Processing accepted loans chu

In [8]:
db_filename = 'loans.db'
conn = sqlite3.connect(db_filename)

accepted_sql = "SELECT * FROM bnpl_like_accepted_loans"
chunk_size = 10000

# Dictionary to accumulate frequency counts for each categorical column.
categorical_counts = {}

# Process the data in chunks.
for i, chunk in enumerate(pd.read_sql_query(accepted_sql, conn, chunksize=chunk_size)):
    print(f"Processing categorical chunk {i+1}")
    
    # Identify categorical columns.
    # You might need to adjust the selection depending on how your columns are stored.
    categorical_cols = chunk.select_dtypes(include=['object', 'category']).columns.tolist()
    
    for col in categorical_cols:
        # Get frequency counts for this chunk (including NaNs if desired)
        counts = chunk[col].value_counts(dropna=False)
        
        # Update the running counts.
        for category, count in counts.items():
            # Initialize the column in the dictionary if needed.
            if col not in categorical_counts:
                categorical_counts[col] = {}
            # Update count for the category.
            categorical_counts[col][category] = categorical_counts[col].get(category, 0) + count

conn.close()

# Now, compile a summary DataFrame for categorical columns.
summary_list = []
for col, counts_dict in categorical_counts.items():
    # Convert the counts dictionary to a Series for easier analysis.
    freq_series = pd.Series(counts_dict)
    
    total_count = freq_series.sum()
    unique_count = freq_series.count()
    mode_value = freq_series.idxmax()  # Category with the highest frequency
    mode_count = freq_series.max()
    
    # Create a DataFrame row for this column.
    summary_list.append({
        'column': col,
        'total_count': total_count,
        'unique_categories': unique_count,
        'mode': mode_value,
        'mode_count': mode_count
    })

categorical_summary_df = pd.DataFrame(summary_list)
categorical_summary_df.to_csv("bnpl_like_accepted_loans_categorical_summary.csv", index=False)

print("Overall categorical summary statistics saved to 'bnpl_like_accepted_loans_categorical_summary.csv'.")

Processing categorical chunk 1
Processing categorical chunk 2
Processing categorical chunk 3
Processing categorical chunk 4
Processing categorical chunk 5
Processing categorical chunk 6
Processing categorical chunk 7
Processing categorical chunk 8
Processing categorical chunk 9
Processing categorical chunk 10
Processing categorical chunk 11
Processing categorical chunk 12
Processing categorical chunk 13
Processing categorical chunk 14
Processing categorical chunk 15
Processing categorical chunk 16
Processing categorical chunk 17
Processing categorical chunk 18
Processing categorical chunk 19
Processing categorical chunk 20
Processing categorical chunk 21
Processing categorical chunk 22
Processing categorical chunk 23
Processing categorical chunk 24
Processing categorical chunk 25
Processing categorical chunk 26
Processing categorical chunk 27
Processing categorical chunk 28
Processing categorical chunk 29
Processing categorical chunk 30
Processing categorical chunk 31
Processing catego

In [9]:
db_filename = 'loans.db'
conn = sqlite3.connect(db_filename)

accepted_sql = "SELECT * FROM cleaned_accepted_loans"
chunk_size = 10000

# Dictionary to hold running aggregates for each numeric column.
numeric_summary = {}
first_chunk = True

# Process the data in chunks.
for i, chunk in enumerate(pd.read_sql_query(accepted_sql, conn, chunksize=chunk_size)):
    print(f"Processing accepted loans chunk {i+1}")
    
    # Get the list of numeric columns in this chunk.
    numeric_cols = chunk.select_dtypes(include=[np.number]).columns.tolist()
    
    for col in numeric_cols:
        # Drop missing values for this column.
        col_data = chunk[col].dropna()
        
        if col_data.empty:
            continue  # Skip if no valid data in this chunk for the column.
        
        # For the first chunk, initialize the aggregator.
        if first_chunk or col not in numeric_summary:
            numeric_summary[col] = {
                'count': col_data.count(),
                'sum': col_data.sum(),
                'sum_sq': (col_data ** 2).sum(),
                'min': col_data.min(),
                'max': col_data.max()
            }
        else:
            numeric_summary[col]['count'] += col_data.count()
            numeric_summary[col]['sum'] += col_data.sum()
            numeric_summary[col]['sum_sq'] += (col_data ** 2).sum()
            numeric_summary[col]['min'] = min(numeric_summary[col]['min'], col_data.min())
            numeric_summary[col]['max'] = max(numeric_summary[col]['max'], col_data.max())
    
    first_chunk = False

# After processing all chunks, compute overall summary statistics.
overall_stats = {}
for col, stats in numeric_summary.items():
    count = stats['count']
    if count > 0:
        mean = stats['sum'] / count
        # Variance formula: Var = (sum_sq - (sum^2)/count) / (count - 1)
        variance = (stats['sum_sq'] - (stats['sum'] ** 2) / count) / (count - 1) if count > 1 else np.nan
        std = np.sqrt(variance) if variance >= 0 else np.nan
    else:
        mean = np.nan
        std = np.nan
    overall_stats[col] = {
        'count': count,
        'mean': mean,
        'std': std,
        'min': stats['min'],
        'max': stats['max']
    }

# Convert the overall statistics to a DataFrame.
overall_stats_df = pd.DataFrame(overall_stats).transpose()

# Save the summary statistics to CSV.
overall_stats_df.to_csv("cleaned_accepted_loans_numeric_summary.csv")

print("Overall numeric summary statistics saved to 'cleaned_accepted_loans_numeric_summary.csv'.")
conn.close()

Processing accepted loans chunk 1
Processing accepted loans chunk 2
Processing accepted loans chunk 3
Processing accepted loans chunk 4
Processing accepted loans chunk 5
Processing accepted loans chunk 6
Processing accepted loans chunk 7
Processing accepted loans chunk 8
Processing accepted loans chunk 9
Processing accepted loans chunk 10
Processing accepted loans chunk 11
Processing accepted loans chunk 12
Processing accepted loans chunk 13
Processing accepted loans chunk 14
Processing accepted loans chunk 15
Processing accepted loans chunk 16
Processing accepted loans chunk 17
Processing accepted loans chunk 18
Processing accepted loans chunk 19
Processing accepted loans chunk 20
Processing accepted loans chunk 21
Processing accepted loans chunk 22
Processing accepted loans chunk 23
Processing accepted loans chunk 24
Processing accepted loans chunk 25
Processing accepted loans chunk 26
Processing accepted loans chunk 27
Processing accepted loans chunk 28
Processing accepted loans chu

In [10]:
db_filename = 'loans.db'
conn = sqlite3.connect(db_filename)

accepted_sql = "SELECT * FROM cleaned_accepted_loans"
chunk_size = 10000

# Dictionary to accumulate frequency counts for each categorical column.
categorical_counts = {}

# Process the data in chunks.
for i, chunk in enumerate(pd.read_sql_query(accepted_sql, conn, chunksize=chunk_size)):
    print(f"Processing categorical chunk {i+1}")
    
    # Identify categorical columns.
    # You might need to adjust the selection depending on how your columns are stored.
    categorical_cols = chunk.select_dtypes(include=['object', 'category']).columns.tolist()
    
    for col in categorical_cols:
        # Get frequency counts for this chunk (including NaNs if desired)
        counts = chunk[col].value_counts(dropna=False)
        
        # Update the running counts.
        for category, count in counts.items():
            # Initialize the column in the dictionary if needed.
            if col not in categorical_counts:
                categorical_counts[col] = {}
            # Update count for the category.
            categorical_counts[col][category] = categorical_counts[col].get(category, 0) + count

conn.close()

# Now, compile a summary DataFrame for categorical columns.
summary_list = []
for col, counts_dict in categorical_counts.items():
    # Convert the counts dictionary to a Series for easier analysis.
    freq_series = pd.Series(counts_dict)
    
    total_count = freq_series.sum()
    unique_count = freq_series.count()
    mode_value = freq_series.idxmax()  # Category with the highest frequency
    mode_count = freq_series.max()
    
    # Create a DataFrame row for this column.
    summary_list.append({
        'column': col,
        'total_count': total_count,
        'unique_categories': unique_count,
        'mode': mode_value,
        'mode_count': mode_count
    })

categorical_summary_df = pd.DataFrame(summary_list)
categorical_summary_df.to_csv("cleaned_accepted_loans_categorical_summary.csv", index=False)

print("Overall categorical summary statistics saved to 'cleaned_accepted_loans_categorical_summary.csv'.")

Processing categorical chunk 1
Processing categorical chunk 2
Processing categorical chunk 3
Processing categorical chunk 4
Processing categorical chunk 5
Processing categorical chunk 6
Processing categorical chunk 7
Processing categorical chunk 8
Processing categorical chunk 9
Processing categorical chunk 10
Processing categorical chunk 11
Processing categorical chunk 12
Processing categorical chunk 13
Processing categorical chunk 14
Processing categorical chunk 15
Processing categorical chunk 16
Processing categorical chunk 17
Processing categorical chunk 18
Processing categorical chunk 19
Processing categorical chunk 20
Processing categorical chunk 21
Processing categorical chunk 22
Processing categorical chunk 23
Processing categorical chunk 24
Processing categorical chunk 25
Processing categorical chunk 26
Processing categorical chunk 27
Processing categorical chunk 28
Processing categorical chunk 29
Processing categorical chunk 30
Processing categorical chunk 31
Processing catego

In [ ]:
import os
import sqlite3
import pandas as pd


# Define the database filename
db_filename = 'loans.db'

# Connect to the SQLite database
conn = sqlite3.connect(db_filename)

# Load the BNPL-like accepted loans table into a DataFrame
df = pd.read_sql_query("SELECT * FROM bnpl_like_accepted_loans_limited", conn)

# Ensure that the date columns are in datetime format
date_cols = ['issue_d', 'earliest_cr_line', 'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Create an approximate age variable.
# Assumption: Borrowers are assumed to have started credit at age 18.
# So, approximate age = (difference in years between issue_d and earliest_cr_line) + 18
df['approx_age'] = (df['issue_d'] - df['earliest_cr_line']).dt.days / 365.25 + 18

# Clip the approximate age to be within a plausible range (between 18 and 65)
df['approx_age'] = df['approx_age'].clip(lower=18, upper=65)

# Output summary statistics for the new age variable
age_summary = df['approx_age'].describe()
print("Summary statistics for the approximate age variable:")
print(age_summary)

# Save the updated BNPL-like data with the age column into a new table in the database
df.to_sql("bnpl_like_accepted_loans_limited_with_age", conn, if_exists="replace", index=False)
print("Updated BNPL-like accepted loans with age saved into the database as 'bnpl_like_accepted_loans_limited_with_age'.")

conn.close()


/var/folders/bm/0wq19v017y79n8fk7_72tt4c0000gn/T/ipykernel_47476/510138796.py:21: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')


Summary statistics for the approximate age variable:
count    184190.000000
mean         33.071414
std           7.652347
min          18.503765
25%          28.001369
50%          31.585216
75%          36.833676
max          65.000000
Name: approx_age, dtype: float64


KeyboardInterrupt: 

In [ ]:
import os
import sqlite3
import pandas as pd

# Define the database filename
db_filename = 'loans.db'

# Connect to the SQLite database
conn = sqlite3.connect(db_filename)

# Load the BNPL-like accepted loans table into a DataFrame
df = pd.read_sql_query("SELECT * FROM bnpl_like_accepted_loans_limited", conn)

# Ensure that the date columns are in datetime format
date_cols = ['issue_d', 'earliest_cr_line', 'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Create an approximate age variable.
# Assumption: Borrowers are assumed to have started credit at age 18.
# So, approximate age = (difference in years between issue_d and earliest_cr_line) + 18
df['approx_age'] = (df['issue_d'] - df['earliest_cr_line']).dt.days / 365.25 + 18

# Clip the approximate age to be within a plausible range (between 18 and 65)
df['approx_age'] = df['approx_age'].clip(lower=18, upper=65)

# Output summary statistics for the new age variable
age_summary = df['approx_age'].describe()
print("Summary statistics for the approximate age variable:")
print(age_summary)

# Define the mapping for the new 'default' column based on 'loan_status'
mapping = {
    "Charged Off": "Default",
    "Default": "Default",
    "Does not meet the credit policy. Status:Charged Off": "Default",
    "Late (16-30 days)": "Default",
    "Late (31-120 days)": "Default",
    "Fully Paid": "Fully Paid",
    "In Grace Period": "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid": "Fully Paid"
}

# Create the new column 'default' using the mapping
df['default'] = df['loan_status'].map(mapping)

# Create the default_indicator column:
#  - 1 for Default, 0 for Fully Paid, and NA for Current.
df['default_indicator'] = df['default'].apply(lambda x: 1 if x == "Default" 
                                               else (0 if x == "Fully Paid" 
                                                     else pd.NA))

# Optionally, display counts for the new columns to verify mapping
print("\nDefault column counts:")
print(df['default'].value_counts())

print("\nDefault Indicator column counts:")
print(df['default_indicator'].value_counts(dropna=False))

# Save the updated BNPL-like data with the age, default, and default_indicator columns 
# into a new table in the database
df.to_sql("bnpl_like_accepted_loans_limited_with_age", conn, if_exists="replace", index=False)
print("\nUpdated BNPL-like accepted loans with age, default, and default_indicator saved into the database as 'bnpl_like_accepted_loans_limited_with_age'.")

conn.close()


/var/folders/bm/0wq19v017y79n8fk7_72tt4c0000gn/T/ipykernel_70383/3013388671.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')


Summary statistics for the approximate age variable:
count    297269.000000
mean         33.148721
std           7.790703
min          18.503765
25%          28.080767
50%          31.661875
75%          36.918549
max          65.000000
Name: approx_age, dtype: float64

Default column counts:
default
Fully Paid    153189
Current       113079
Default        31023
Name: count, dtype: int64

Default Indicator column counts:
default_indicator
0       153189
<NA>    113079
1        31023
Name: count, dtype: int64

Updated BNPL-like accepted loans with age, default, and default_indicator saved into the database as 'bnpl_like_accepted_loans_limited_with_age'.


In [12]:
from fredapi import Fred
import pandas as pd
import ssl

ssl._create_default_https_context = ssl._create_stdlib_context

# Replace with your actual FRED API key.
fred = Fred(api_key='8b4509f14eefacaa31c5bd7b1bd1723b')

# Define the time range for the macro data.
start_date = '2005-01-01'
end_date = '2020-12-31'

# --- Download Macro Variables ---

# GDP (Gross Domestic Product) – take the quarterly end-of-quarter value.
gdp = fred.get_series('GDP', observation_start=start_date, observation_end=end_date)
gdp = gdp.resample('Q').last()

# Compute GDP growth as the year-over-year percentage change (4-quarter change).
gdp_growth = gdp.pct_change(periods=4) * 100
gdp_growth = gdp_growth.dropna()  # Drop initial NaN values

# Unemployment Rate – take the quarterly average.
unrate = fred.get_series('UNRATE', observation_start=start_date, observation_end=end_date)
unrate = unrate.resample('Q').mean()

# Federal Funds Rate – take the quarterly average.
fedfunds = fred.get_series('FEDFUNDS', observation_start=start_date, observation_end=end_date)
fedfunds = fedfunds.resample('Q').mean()

# CPI – use the CPI for All Urban Consumers (CPIAUCSL) to compute inflation.
cpi = fred.get_series('CPIAUCSL', observation_start=start_date, observation_end=end_date)
cpi = cpi.resample('Q').last()

# Compute annualized inflation as the 4-quarter percentage change in CPI.
inflation = cpi.pct_change(periods=4) * 100
inflation = inflation.dropna()

# Housing Prices – using the Case‑Shiller U.S. National Home Price Index (CSUSHPINSA).
housing = fred.get_series('CSUSHPINSA', observation_start=start_date, observation_end=end_date)
housing = housing.resample('Q').last()

# --- Align Data into a Single DataFrame ---

# Use GDP's index as the base quarterly timeline.
macro_df = pd.DataFrame({
    'GDP': gdp,
    'gdp_growth': gdp_growth,
    'unemployment_rate': unrate,
    'fedfunds': fedfunds,
    'housing_price': housing
})

# Join the inflation series.
macro_df = macro_df.join(inflation.rename('inflation'))

# --- Format the Quarter Identifier ---

# Convert the DatetimeIndex to a PeriodIndex in the format "YYYYQX"
macro_df.index = macro_df.index.to_period('Q').astype(str)
macro_df.reset_index(inplace=True)
macro_df.rename(columns={'index': 'quarter'}, inplace=True)

# Save the macro data (with housing prices) to CSV.
macro_df.to_csv('macro_data.csv', index=False)
print("Macro data with GDP growth and housing prices saved to macro_data.csv")
print(macro_df.head())

/var/folders/bm/0wq19v017y79n8fk7_72tt4c0000gn/T/ipykernel_47476/3131548405.py:18: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  gdp = gdp.resample('Q').last()
/var/folders/bm/0wq19v017y79n8fk7_72tt4c0000gn/T/ipykernel_47476/3131548405.py:26: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  unrate = unrate.resample('Q').mean()
/var/folders/bm/0wq19v017y79n8fk7_72tt4c0000gn/T/ipykernel_47476/3131548405.py:30: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  fedfunds = fedfunds.resample('Q').mean()
/var/folders/bm/0wq19v017y79n8fk7_72tt4c0000gn/T/ipykernel_47476/3131548405.py:34: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  cpi = cpi.resample('Q').last()


Macro data with GDP growth and housing prices saved to macro_data.csv
  quarter        GDP  gdp_growth  unemployment_rate  fedfunds  housing_price  \
0  2005Q1  12767.286         NaN           5.300000  2.470000        164.577   
1  2005Q2  12922.656         NaN           5.100000  2.943333        172.017   
2  2005Q3  13142.642         NaN           4.966667  3.460000        177.612   
3  2005Q4  13324.204         NaN           4.966667  3.980000        180.108   
4  2006Q1  13599.160    6.515668           4.733333  4.456667        182.749   

   inflation  
0        NaN  
1        NaN  
2        NaN  
3        NaN  
4   3.417918  


/var/folders/bm/0wq19v017y79n8fk7_72tt4c0000gn/T/ipykernel_47476/3131548405.py:42: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  housing = housing.resample('Q').last()


In [2]:
# ------------------------------
# ADDITIONAL CODE TO MAKE DATA NON-STATIC
# ------------------------------

import numpy as np

def create_time_slices(row):
    slices = {}
    
    # --- Determine the Default Slice ---
    # For borrowers with default_indicator == 1, randomly choose a default slice between 1 and 4.
    if row['default_indicator'] == 1:
        default_slice = np.random.randint(1, 5)  # Random integer between 1 and 4.
    else:
        default_slice = None  # Non-defaulters have no default slice.
    
    # --- Total Payment (total_pymnt) ---
    total_payment = row['total_pymnt'] if not pd.isnull(row['total_pymnt']) else 0
    for t in range(1, 5):
        if default_slice and t >= default_slice:
            # For defaulters, freeze payment progression after the default slice.
            progress = (default_slice - 1) / 4 if default_slice > 1 else 0
            slices[f"total_pymnt_t{t}"] = total_payment * progress
        else:
            slices[f"total_pymnt_t{t}"] = total_payment * (t / 4)
    
    # --- Revolving Balance (revol_bal) ---
    initial_revol_bal = row['revol_bal'] if not pd.isnull(row['revol_bal']) else 0
    for t in range(1, 5):
        if default_slice and t >= default_slice:
            progress = (default_slice - 1) / 4 if default_slice > 1 else 0
            slices[f"revol_bal_t{t}"] = initial_revol_bal * (1 - progress)
        else:
            slices[f"revol_bal_t{t}"] = initial_revol_bal * (1 - t / 4)
    
    # --- Last Payment Amount (last_pymnt_amnt) ---
    per_payment = total_payment / 4 if total_payment else 0
    for t in range(1, 5):
        if default_slice:
            if t == default_slice:
                slices[f"last_pymnt_amnt_t{t}"] = row['last_pymnt_amnt'] if not pd.isnull(row['last_pymnt_amnt']) else per_payment
            elif t > default_slice:
                slices[f"last_pymnt_amnt_t{t}"] = 0
            else:
                slices[f"last_pymnt_amnt_t{t}"] = per_payment
        else:
            slices[f"last_pymnt_amnt_t{t}"] = per_payment
    
    # --- Default Indicator per Time Slice ---
    for t in range(1, 5):
        slices[f"default_indicator_t{t}"] = 1 if (default_slice and t >= default_slice) else 0

    # --- Dynamic FICO Score using fico_range_high ---
    # Base FICO score is taken from fico_range_high.
    if not pd.isnull(row['fico_range_high']):
        base_fico = row['fico_range_high']
    else:
        base_fico = np.nan

    # Set the maximum total improvement/decline to 100 points.
    # Cap improvement so that fico does not exceed 850 and decline so that it does not drop below 300.
    if not np.isnan(base_fico):
        max_possible_improvement = 100 if base_fico <= 750 else (850 - base_fico)
        max_possible_decline = 100 if base_fico >= 300 else (base_fico - 300)
    else:
        max_possible_improvement = 0
        max_possible_decline = 0

    total_improvement = np.random.uniform(0, max_possible_improvement)
    total_decline = np.random.uniform(0, max_possible_decline)
    
    if default_slice is None:
        # For non-defaulters, increase FICO steadily over 6 slices.
        for t in range(1, 5):
            new_score = base_fico + total_improvement * (t / 4)
            slices[f"fico_score_t{t}"] = min(new_score, 850)
    else:
        # For defaulters, FICO remains stable until default, then declines linearly.
        for t in range(1, default_slice):
            new_score = base_fico + total_improvement * (t / 4)
            slices[f"fico_score_t{t}"] = min(new_score, 850)
        remaining_periods = 5 - default_slice  # Includes the default slice.
        for t in range(default_slice, 5):
            decline_fraction = (t - default_slice + 1) / remaining_periods
            new_score = base_fico - total_decline * decline_fraction
            slices[f"fico_score_t{t}"] = max(new_score, 300)
    
    # --- Dynamic Revolving Utilization (revol_util) ---
    # Calculate utilization as dynamic revol_bal divided by static total_rev_hi_lim.
    credit_limit = row['total_rev_hi_lim'] if (not pd.isnull(row.get('total_rev_hi_lim', np.nan)) and row.get('total_rev_hi_lim', 0) != 0) else np.nan
    for t in range(1, 5):
        if not np.isnan(credit_limit):
            util = slices[f"revol_bal_t{t}"] / credit_limit
            slices[f"revol_util_t{t}"] = min(max(util, 0), 1)
        else:
            slices[f"revol_util_t{t}"] = np.nan

    # --- Dynamic Outstanding Principal (out_prncp) ---
    # Use the static out_prncp as the base outstanding principal.
    base_out_prncp = row['out_prncp'] if not pd.isnull(row['out_prncp']) else 0
    for t in range(1, 5):
        if default_slice and t >= default_slice:
            progress = (default_slice - 1) / 4 if default_slice > 1 else 0
            slices[f"out_prncp_t{t}"] = base_out_prncp * (1 - progress)
        else:
            slices[f"out_prncp_t{t}"] = base_out_prncp * (1 - t / 4)
    
    # --- Dynamic Delinquency Count (delinq_count) ---
    # For non-defaulters, assume no new delinquencies.
    # For defaulters, assume a delinquency event occurs at the default slice and increases thereafter.
    for t in range(1, 5):
        if default_slice:
            if t < default_slice:
                slices[f"delinq_count_t{t}"] = 0
            else:
                slices[f"delinq_count_t{t}"] = t - default_slice + 1
        else:
            slices[f"delinq_count_t{t}"] = 0

    return pd.Series(slices)

In [3]:
import os
import sqlite3
import pandas as pd
import math

# Define the database filename
db_filename = 'loans.db'

# Connect to the SQLite database
conn = sqlite3.connect(db_filename)

# Load the BNPL-like accepted loans table into a DataFrame
df = pd.read_sql_query("SELECT * FROM bnpl_like_accepted_loans_limited", conn)

# Ensure that the date columns are in datetime format
date_cols = ['issue_d', 'earliest_cr_line', 'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Create an approximate age variable.
# Assumption: Borrowers are assumed to have started credit at age 18.
# So, approximate age = (difference in years between issue_d and earliest_cr_line) + 18
df['approx_age'] = (df['issue_d'] - df['earliest_cr_line']).dt.days / 365.25 + 18

# Clip the approximate age to be within a plausible range (between 18 and 65)
df['approx_age'] = df['approx_age'].clip(lower=18, upper=65)

# Output summary statistics for the new age variable
age_summary = df['approx_age'].describe()
print("Summary statistics for the approximate age variable:")
print(age_summary)

# Define the mapping for the new 'default' column based on 'loan_status'
mapping = {
    "Charged Off": "Default",
    "Default": "Default",
    "Does not meet the credit policy. Status:Charged Off": "Default",
    "Late (16-30 days)": "Default",
    "Late (31-120 days)": "Default",
    "Current": "Current",
    "Fully Paid": "Fully Paid",
    "In Grace Period": "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid": "Fully Paid"
}

# Create the new column 'default' using the mapping
df['default'] = df['loan_status'].map(mapping)

# Create the default_indicator column:
#  - 1 for Default, 0 for Fully Paid, and NA for Current.
df['default_indicator'] = df['default'].apply(lambda x: 1 if x == "Default" 
                                               else (0 if x == "Fully Paid" 
                                                     else pd.NA))

# --- New Section: FICO Score Buckets and Scaling using fico_range_high ---

# Define a function to assign the standard FICO bucket based on fixed score ranges.
def get_fico_bucket(score):
    if pd.isna(score):
        return pd.NA
    if 300 <= score <= 579:
        return "deep subprime"
    elif 580 <= score <= 619:
        return "subprime"
    elif 620 <= score <= 659:
        return "near prime"
    elif 660 <= score <= 719:
        return "prime"
    elif 720 <= score <= 850:
        return "super-prime"
    else:
        return pd.NA

# Create the fico_bucket column by applying the function to the fico_range_high column.

# Scale the raw fico_range_high to the [300, 850] range using min-max scaling.
min_score = df['fico_range_high'].min()
max_score = df['fico_range_high'].max()
df['scaled_fico'] = 300 + (df['fico_range_high'] - min_score) / (max_score - min_score) * (850 - 300)

df['fico_bucket'] = df['scaled_fico'].apply(get_fico_bucket)

# --- New Section: Bucketing FICO Scores by Percentage Breakdown ---

# Compute quantile thresholds for the desired percentages (for rows with a fico_range_high value)
q1 = df['fico_range_high'].quantile(0.489)  # 48.9%
q2 = df['fico_range_high'].quantile(0.649)  # 48.9% + 16.0% = 64.9%
q3 = df['fico_range_high'].quantile(0.776)  # 64.9% + 12.7% = 77.6%
q4 = df['fico_range_high'].quantile(0.908)  # 77.6% + 13.2% = 90.8%

def get_fico_bucket_pct(score):

    # For missing scores, return a dedicated bucket label.
    if pd.isna(score):
        return "No Score"
    # Assign bucket based on the quantile thresholds.
    if score < q1:
        return "Deep Subprime"
    elif score < q2:
        return "Subprime"
    elif score < q3:
        return "Near Prime"
    elif score < q4:
        return "Prime"
    else:
        return "Super-prime"
    
time_slices = df.apply(create_time_slices, axis=1)
df = pd.concat([df, time_slices], axis=1)

# Create the new column using the above function
df['fico_bucket_pct'] = df['fico_range_high'].apply(get_fico_bucket_pct)
df['fico_bucket_pct_t0'] = df['fico_range_high'].apply(get_fico_bucket_pct)
df['fico_bucket_pct_t1'] = df['fico_score_t1'].apply(get_fico_bucket_pct)
df['fico_bucket_pct_t2'] = df['fico_score_t2'].apply(get_fico_bucket_pct)
df['fico_bucket_pct_t3'] = df['fico_score_t3'].apply(get_fico_bucket_pct)
df['fico_bucket_pct_t4'] = df['fico_score_t4'].apply(get_fico_bucket_pct)



# Optionally, display counts for the new columns to verify the mappings.
print("\nDefault column counts:")
print(df['default'].value_counts())

print("\nDefault Indicator column counts:")
print(df['default_indicator'].value_counts(dropna=False))

print("\nFICO Bucket counts (standard ranges):")
print(df['fico_bucket'].value_counts(dropna=False))

print("\nScaled FICO scores (first few rows):")
print(df[['fico_range_high', 'scaled_fico', 'fico_bucket']].head())

print("\nFICO Bucket by Percentage Breakdown counts:")
print(df['fico_bucket_pct'].value_counts(dropna=False))

# Define a helper function to shift a quarter string by n quarters.
def shift_quarter(q, n):
    """
    Shifts a quarter string by n quarters.
    q: a string in the format "YYYYQX", e.g. "2015Q3"
    n: number of quarters to shift (can be negative)
    Returns a new quarter string.
    """
    year = int(q[:4])
    qtr = int(q[-1])
    # Convert to total quarters (0-indexed)
    total_quarters = (year * 4) + (qtr - 1) + n  
    new_year = total_quarters // 4
    new_qtr = (total_quarters % 4) + 1
    return f"{new_year}Q{new_qtr}"

# For each of the 6 time slices, create a temporary column representing the macro quarter.
# Mapping: t1 = shift -1, t2 = shift 0, t3 = shift +1, t4 = shift +2, t5 = shift +3, t6 = shift +4.
for t in range(1, 6):
    shift_val = -1 + (t - 1)  # Produces: -1, 0, 1, 2, 3, 4 respectively.
    df[f"macro_quarter_t{t}"] = df['issue_quarter'].apply(lambda q: shift_quarter(q, shift_val))

# Load the macroeconomic data from the CSV.
macro_df = pd.read_csv('macro_data.csv')
# Ensure the quarter column is treated as a string.
macro_df['quarter'] = macro_df['quarter'].astype(str)

# List of macro variables from your macro_data.csv.
macro_vars = ['GDP', 'gdp_growth', 'unemployment_rate', 'fedfunds', 'inflation', 'housing_price']

# For each time slice, merge the macro variables into df.
for t in range(1, 6):
    # Merge using the temporary macro_quarter column.
    df = df.merge(
        macro_df[['quarter'] + macro_vars],
        left_on=f"macro_quarter_t{t}",
        right_on="quarter",
        how="left",
        suffixes=("", f"_t{t}")
    )
    # Rename each macro variable to include the time slice suffix.
    for var in macro_vars:
        df.rename(columns={var: f"{var}_t{t}"}, inplace=True)
    # Drop the extra 'quarter' column from the merge.
    df.drop(columns=["quarter"], inplace=True)

# Remove the temporary macro_quarter columns.
for t in range(1, 6):
    df.drop(columns=[f"macro_quarter_t{t}"], inplace=True)

# (Optional) Preview a few rows to verify the new macro columns.
macro_columns = [f"{var}_t{t}" for var in macro_vars for t in range(1, 6)]
print("\nPreview of macroeconomic variables by time slice:")
print(df[['issue_quarter'] + macro_columns].drop_duplicates().sort_values('issue_quarter').head())


# Save the updated BNPL-like data with all new columns into a new table in the database
df.to_sql("bnpl_like_accepted_loans_limited_with_age_fico", conn, if_exists="replace", index=False)
print("\nUpdated BNPL-like accepted loans with age, default, default_indicator, fico_bucket, scaled_fico, and fico_bucket_pct saved into the database as 'bnpl_like_accepted_loans_limited_with_age_fico'.")

conn.close()


/var/folders/bm/0wq19v017y79n8fk7_72tt4c0000gn/T/ipykernel_46515/2687411867.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')


Summary statistics for the approximate age variable:
count    184190.000000
mean         33.071414
std           7.652347
min          18.503765
25%          28.001369
50%          31.585216
75%          36.833676
max          65.000000
Name: approx_age, dtype: float64

Default column counts:
default
Fully Paid    153189
Default        31023
Name: count, dtype: int64

Default Indicator column counts:
default_indicator
0    153189
1     31023
Name: count, dtype: int64

FICO Bucket counts (standard ranges):
fico_bucket
deep subprime    157376
subprime           8008
prime              5916
near prime         5532
<NA>               4410
super-prime        2970
Name: count, dtype: int64

Scaled FICO scores (first few rows):
   fico_range_high  scaled_fico    fico_bucket
0            679.0   451.483051  deep subprime
1            704.0   509.745763  deep subprime
2            669.0   428.177966  deep subprime
3            729.0   568.008475  deep subprime
4            674.0   439.830508  d

In [4]:
import os
import sqlite3
import pandas as pd

# Define the database filename
db_filename = 'loans.db'
conn = sqlite3.connect(db_filename)

# Load the BNPL-like accepted loans (with age already computed) into a DataFrame
df = pd.read_sql_query("SELECT * FROM bnpl_like_accepted_loans_limited_with_age_fico", conn)

# Define age bins and labels (adjust boundaries as needed)
bins = [18, 24, 33, 40, 50, 64, 65]
labels = ["18-24", "25-33", "34-40", "41-50", "51-64", "65"]

# Create a new column for age groups using pd.cut
df['age_group'] = pd.cut(df['approx_age'], bins=bins, labels=labels, right=True, include_lowest=True)

# Calculate the counts for each age group
age_group_counts = df['age_group'].value_counts().sort_index()
print("Counts for each age group:")
print(age_group_counts)

df.to_csv("rebalanced_age_data_original_100FICO.csv", index=False)

'''
# Calculate the relative percentage for each age group
total = age_group_counts.sum()
age_group_percent = (age_group_counts / total * 100).round(2)
print("Relative percent for each age group:")
print(age_group_percent)

# Combine counts and percentages into a summary DataFrame
age_group_summary = pd.DataFrame({'count': age_group_counts, 'percent': age_group_percent})

# Save the summary DataFrame to a CSV file
age_group_summary.to_csv("age_group_summary.csv", index_label="age_group")

conn.close()
'''


Counts for each age group:
age_group
18-24    16262
25-33    91581
34-40    46788
41-50    23007
51-64     6045
65         507
Name: count, dtype: int64


'\n# Calculate the relative percentage for each age group\ntotal = age_group_counts.sum()\nage_group_percent = (age_group_counts / total * 100).round(2)\nprint("Relative percent for each age group:")\nprint(age_group_percent)\n\n# Combine counts and percentages into a summary DataFrame\nage_group_summary = pd.DataFrame({\'count\': age_group_counts, \'percent\': age_group_percent})\n\n# Save the summary DataFrame to a CSV file\nage_group_summary.to_csv("age_group_summary.csv", index_label="age_group")\n\nconn.close()\n'

In [6]:
import os
import sqlite3
import pandas as pd
import numpy as np

# For reproducibility
np.random.seed(42)

# Define the database filename and load the data
db_filename = 'loans.db'
conn = sqlite3.connect(db_filename)
df = pd.read_sql_query("SELECT * FROM bnpl_like_accepted_loans_limited_with_age_fico", conn)
conn.close()

# Define age bins and labels – these are the original bins
bins = [18, 24, 33, 40, 50, 64, 100]  # extend the last bin so that 65+ is grouped (assume max age < 100)
labels = ["18-24", "25-33", "34-40", "41-50", "51-64", "65+"]
df['age_group'] = pd.cut(df['approx_age'], bins=bins, labels=labels, right=True, include_lowest=True)

# Calculate the current counts for each age group
current_counts = df['age_group'].value_counts().sort_index()
print("Current counts for each age group:")
print(current_counts)

# We will only rebalance rows with age_group != "65+"
non65_mask = df['age_group'] != "65+"
df_non65 = df[non65_mask].copy()

# Total non-65 count:
total_non65 = df_non65.shape[0]

# Desired (raw) target percentages (as given, midpoints of ranges)
# These raw numbers sum to 37.5 + 32.5 + 27.5 + 20 + 10 = 127.5, so we normalize.
raw_targets = {
    "18-24": 30,
    "25-33": 30,
    "34-40": 20,
    "41-50": 10,
    "51-64": 10
}
raw_sum = sum(raw_targets.values())
normalized_targets = {k: v/raw_sum for k, v in raw_targets.items()}

# Calculate target counts for each non-65 bin based on normalized percentages.
target_counts = {bin_label: int(round(normalized_targets[bin_label] * total_non65))
                 for bin_label in raw_targets.keys()}

print("\nTarget counts for non-65 age groups (normalized):")
print(target_counts)

# Get current counts for non-65 bins (as a dict)
current_non65_counts = df_non65['age_group'].value_counts().to_dict()
print("\nCurrent non-65 counts:")
print(current_non65_counts)

# Compute deficits (if target is higher than current) and surpluses (if target is lower than current)
deficits = {}
surpluses = {}
for group in raw_targets.keys():
    current = current_non65_counts.get(group, 0)
    target = target_counts[group]
    if current < target:
        deficits[group] = target - current
    elif current > target:
        surpluses[group] = current - target

print("\nDeficits (need additional rows):")
print(deficits)
print("\nSurpluses (rows to reassign):")
print(surpluses)

# Define a helper mapping for age ranges corresponding to each age group.
# Here we assume integer ages uniformly distributed in the range.
age_range_mapping = {
    "18-24": (18, 24),
    "25-33": (25, 33),
    "34-40": (34, 40),
    "41-50": (41, 50),
    "51-64": (51, 64)
}

# Function to randomly sample a new age from the given range (inclusive)
def sample_new_age(age_range):
    return np.random.randint(age_range[0], age_range[1] + 1)

# We will reassign rows from surplus groups to deficit groups.
# For each deficit group, we will pull rows from the available surplus bins.
# (A more sophisticated allocation might reassign proportionally;
# here we iterate over deficits and try to fill each deficit from available surplus.)

# Work on a copy of df_non65 to update rows
df_rebalanced = df_non65.copy()

# For each deficit group, fill the deficit by taking rows from surplus groups.
for target_group, deficit in deficits.items():
    print(f"\nFilling deficit for {target_group}: need {deficit} rows")
    remaining_deficit = deficit
    # Iterate over surplus groups (order can be chosen arbitrarily; here we loop over surpluses)
    for source_group in list(surpluses.keys()):
        available = surpluses[source_group]
        if available <= 0:
            continue
        # Determine how many rows to move from this surplus bin to the target
        num_to_move = min(available, remaining_deficit)
        print(f"  Reassigning {num_to_move} rows from {source_group} to {target_group}")
        # Identify candidate rows in source_group
        candidates = df_rebalanced[df_rebalanced['age_group'] == source_group]
        # Randomly sample indices to reassign
        indices_to_reassign = candidates.sample(n=num_to_move, random_state=42).index
        # Update these rows: assign a new 'approx_age' in the target group's range
        new_age_range = age_range_mapping[target_group]
        new_ages = [sample_new_age(new_age_range) for _ in range(num_to_move)]
        df_rebalanced.loc[indices_to_reassign, 'approx_age'] = new_ages
        # Also update the age_group column for these rows
        df_rebalanced.loc[indices_to_reassign, 'age_group'] = target_group
        
        # Update counters: decrease surplus for source_group and decrease remaining deficit
        surpluses[source_group] -= num_to_move
        remaining_deficit -= num_to_move
        if remaining_deficit <= 0:
            break
    if remaining_deficit > 0:
        print(f"Warning: Could not fill entire deficit for {target_group}. {remaining_deficit} rows remain unfilled.")

# Now, combine the rebalanced non-65 rows with the 65+ rows (which were not changed)
df_final = pd.concat([df_rebalanced, df[df['age_group'] == "65+"]])

# Recalculate final counts for non-65 groups
final_non65_counts = df_final[df_final['age_group'] != "65+"]['age_group'].value_counts().sort_index()
print("\nFinal non-65 counts after rebalancing:")
print(final_non65_counts)

# (Optional) Save the rebalanced dataset to a new CSV file
df_final.to_csv("rebalanced_age_data.csv", index=False)
print("\nRebalanced data saved to 'rebalanced_age_data.csv'.")


Current counts for each age group:
age_group
18-24    16262
25-33    91581
34-40    46788
41-50    23007
51-64     6045
65+        507
Name: count, dtype: int64

Target counts for non-65 age groups (normalized):
{'18-24': 55112, '25-33': 55112, '34-40': 36741, '41-50': 18370, '51-64': 18370}

Current non-65 counts:
{'25-33': 91581, '34-40': 46788, '41-50': 23007, '18-24': 16262, '51-64': 6045, '65+': 0}

Deficits (need additional rows):
{'18-24': 38850, '51-64': 12325}

Surpluses (rows to reassign):
{'25-33': 36469, '34-40': 10047, '41-50': 4637}

Filling deficit for 18-24: need 38850 rows
  Reassigning 36469 rows from 25-33 to 18-24
  Reassigning 2381 rows from 34-40 to 18-24

Filling deficit for 51-64: need 12325 rows
  Reassigning 7666 rows from 34-40 to 51-64
  Reassigning 4637 rows from 41-50 to 51-64

Final non-65 counts after rebalancing:
age_group
18-24    55112
25-33    55112
34-40    36741
41-50    18370
51-64    18348
65+          0
Name: count, dtype: int64

Rebalanced data

In [7]:
import pandas as pd
df = pd.read_csv("rebalanced_age_data.csv")
unique_issue_quarters = df['issue_quarter'].unique()
print(unique_issue_quarters)

/var/folders/bm/0wq19v017y79n8fk7_72tt4c0000gn/T/ipykernel_47476/3735270296.py:2: DtypeWarning: Columns (19,49,59,118,129,130,131,134,135,136,139) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("rebalanced_age_data.csv")


['2015Q4' '2015Q3' '2015Q2' '2015Q1' '2018Q1' '2017Q3' '2016Q2' '2018Q3'
 '2017Q2' '2016Q1' '2014Q4' '2014Q3' '2014Q2' '2014Q1' '2018Q4' '2018Q2'
 '2011Q4' '2011Q3' '2011Q2' '2011Q1' '2010Q4' '2010Q3' '2010Q2' '2010Q1'
 '2009Q4' '2009Q3' '2009Q2' '2009Q1' '2008Q4' '2008Q3' '2008Q2' '2008Q1'
 '2007Q4' '2007Q3' '2007Q2' '2017Q1' '2013Q4' '2013Q3' '2013Q2' '2013Q1'
 '2012Q4' '2012Q3' '2012Q2' '2012Q1' '2016Q3' '2017Q4' '2016Q4']


In [5]:
import os
import sqlite3
import pandas as pd
import numpy as np

# For reproducibility
np.random.seed(42)

# Define the database filename and load the data
db_filename = 'loans.db'
conn = sqlite3.connect(db_filename)
df = pd.read_sql_query("SELECT * FROM bnpl_like_accepted_loans_limited_with_age_fico", conn)
conn.close()

# Define age bins and labels – these are the original bins
bins = [18, 24, 33, 40, 50, 64, 100]  # extend the last bin so that 65+ is grouped (assume max age < 100)
labels = ["18-24", "25-33", "34-40", "41-50", "51-64", "65+"]
df['age_group'] = pd.cut(df['approx_age'], bins=bins, labels=labels, right=True, include_lowest=True)

# Calculate the current counts for each age group
current_counts = df['age_group'].value_counts().sort_index()
print("Current counts for each age group:")
print(current_counts)

# We'll only rebalance rows with age_group != "65+"
non65_mask = df['age_group'] != "65+"
df_non65 = df[non65_mask].copy()

# Total non-65 count:
total_non65 = df_non65.shape[0]

# Desired (raw) target percentages (as given, midpoints of ranges)
# These raw numbers sum to 37.5 + 32.5 + 27.5 + 20 + 10 = 127.5, so we normalize.
raw_targets = {
    "18-24": 15,
    "25-33": 30,
    "34-40": 20,
    "41-50": 20,
    "51-64": 15
}
raw_sum = sum(raw_targets.values())
normalized_targets = {k: v / raw_sum for k, v in raw_targets.items()}

# Calculate target counts for each non-65 bin based on normalized percentages.
target_counts = {bin_label: int(round(normalized_targets[bin_label] * total_non65))
                 for bin_label in raw_targets.keys()}

print("\nTarget counts for non-65 age groups (normalized):")
print(target_counts)

# Get current counts for non-65 bins (as a dict)
current_non65_counts = df_non65['age_group'].value_counts().to_dict()
print("\nCurrent non-65 counts:")
print(current_non65_counts)

# Compute deficits (if target is higher than current) and surpluses (if target is lower than current)
deficits = {}
surpluses = {}
for group in raw_targets.keys():
    current = current_non65_counts.get(group, 0)
    target = target_counts[group]
    if current < target:
        deficits[group] = target - current
    elif current > target:
        surpluses[group] = current - target

print("\nDeficits (need additional rows):")
print(deficits)
print("\nSurpluses (rows to reassign):")
print(surpluses)

# Define a helper mapping for age ranges corresponding to each age group.
# Here we assume integer ages uniformly distributed in the range.
age_range_mapping = {
    "18-24": (18, 24),
    "25-33": (25, 33),
    "34-40": (34, 40),
    "41-50": (41, 50),
    "51-64": (51, 64)
}

# Function to randomly sample a new age from the given range (inclusive)
def sample_new_age(age_range):
    return np.random.randint(age_range[0], age_range[1] + 1)

# List to define the sequential order for non-65 bins
ordered_bins = ["18-24", "25-33", "34-40", "41-50", "51-64"]

# Work on a copy of df_non65 to update rows
df_rebalanced = df_non65.copy()

# For each deficit group, only check its immediate neighbors (lower and/or upper)
for target_group in deficits.keys():
    remaining_deficit = deficits[target_group]
    print(f"\nFilling deficit for {target_group}: need {remaining_deficit} rows")
    # Find index of target in ordered list
    target_idx = ordered_bins.index(target_group)
    
    # Identify eligible donor groups (neighbors only)
    eligible_donors = []
    if target_idx - 1 >= 0:
        eligible_donors.append(ordered_bins[target_idx - 1])
    if target_idx + 1 < len(ordered_bins):
        eligible_donors.append(ordered_bins[target_idx + 1])
    
    print(f"  Eligible donor groups for {target_group}: {eligible_donors}")
    
    # Try to fill deficit using eligible donors (first lower then upper)
    for donor_group in eligible_donors:
        # Only proceed if this donor group is in surplus (and if surplus remains)
        donor_surplus = surpluses.get(donor_group, 0)
        if donor_surplus <= 0:
            continue
        
        # Determine how many rows to move from donor_group to target_group
        num_to_move = min(donor_surplus, remaining_deficit)
        print(f"  Reassigning {num_to_move} rows from {donor_group} to {target_group}")
        
        # Identify candidate rows in donor_group
        candidates = df_rebalanced[df_rebalanced['age_group'] == donor_group]
        # Randomly sample indices to reassign
        indices_to_reassign = candidates.sample(n=num_to_move, random_state=42).index
        # Update these rows: assign a new 'approx_age' in the target group's range
        new_age_range = age_range_mapping[target_group]
        new_ages = [sample_new_age(new_age_range) for _ in range(num_to_move)]
        df_rebalanced.loc[indices_to_reassign, 'approx_age'] = new_ages
        # Also update the age_group column for these rows
        df_rebalanced.loc[indices_to_reassign, 'age_group'] = target_group
        
        # Update counters: decrease surplus for donor_group and decrease remaining deficit
        surpluses[donor_group] -= num_to_move
        remaining_deficit -= num_to_move
        if remaining_deficit <= 0:
            break
    
    if remaining_deficit > 0:
        print(f"Warning: Could not fill entire deficit for {target_group}. {remaining_deficit} rows remain unfilled.")

# Now, combine the rebalanced non-65 rows with the 65+ rows (which were not changed)
df_final = pd.concat([df_rebalanced, df[df['age_group'] == "65+"]])

# Recalculate final counts for non-65 groups
final_non65_counts = df_final['age_group'].value_counts().sort_index()
print("\nFinal non-65 counts after rebalancing:")
print(final_non65_counts)

# (Optional) Save the rebalanced dataset to a new CSV file
df_final.to_csv("rebalanced_age_data.csv", index=False)
print("\nRebalanced data saved to 'rebalanced_age_data.csv'.")


Current counts for each age group:
age_group
18-24    16262
25-33    91581
34-40    46788
41-50    23007
51-64     6045
65+        507
Name: count, dtype: int64

Target counts for non-65 age groups (normalized):
{'18-24': 27556, '25-33': 55112, '34-40': 36741, '41-50': 36741, '51-64': 27556}

Current non-65 counts:
{'25-33': 91581, '34-40': 46788, '41-50': 23007, '18-24': 16262, '51-64': 6045, '65+': 0}

Deficits (need additional rows):
{'18-24': 11294, '41-50': 13734, '51-64': 21511}

Surpluses (rows to reassign):
{'25-33': 36469, '34-40': 10047}

Filling deficit for 18-24: need 11294 rows
  Eligible donor groups for 18-24: ['25-33']
  Reassigning 11294 rows from 25-33 to 18-24

Filling deficit for 41-50: need 13734 rows
  Eligible donor groups for 41-50: ['34-40', '51-64']
  Reassigning 10047 rows from 34-40 to 41-50

Filling deficit for 51-64: need 21511 rows
  Eligible donor groups for 51-64: ['41-50']

Final non-65 counts after rebalancing:
age_group
18-24    27556
25-33    80287


In [1]:
import os
import sqlite3
import pandas as pd
import numpy as np

# For reproducibility
np.random.seed(42)

# Define the database filename and load the data
db_filename = 'loans.db'
conn = sqlite3.connect(db_filename)
df = pd.read_sql_query("SELECT * FROM bnpl_like_accepted_loans_limited_with_age_fico", conn)
conn.close()

# Define age bins and labels – these are the original bins
bins = [18, 24, 33, 40, 50, 64, 100]  # extend the last bin so that 65+ is grouped (assume max age < 100)
labels = ["18-24", "25-33", "34-40", "41-50", "51-64", "65+"]
df['age_group'] = pd.cut(df['approx_age'], bins=bins, labels=labels, right=True, include_lowest=True)

# Calculate the current counts for each age group
current_counts = df['age_group'].value_counts().sort_index()
print("Current counts for each age group:")
print(current_counts)

# We will only rebalance rows with age_group != "65+"
non65_mask = df['age_group'] != "65+"
df_non65 = df[non65_mask].copy()

# Total non-65 count:
total_non65 = df_non65.shape[0]

# Desired (raw) target percentages (as given, midpoints of ranges)
# These raw numbers sum to 37.5 + 32.5 + 27.5 + 20 + 10 = 127.5, so we normalize.
raw_targets = {
    "18-24": 25,
    "25-33": 35,
    "34-40": 25,
    "41-50": 10,
    "51-64": 5
}
raw_sum = sum(raw_targets.values())
normalized_targets = {k: v/raw_sum for k, v in raw_targets.items()}

# Calculate target counts for each non-65 bin based on normalized percentages.
target_counts = {bin_label: int(round(normalized_targets[bin_label] * total_non65))
                 for bin_label in raw_targets.keys()}

print("\nTarget counts for non-65 age groups (normalized):")
print(target_counts)

# Get current counts for non-65 bins (as a dict)
current_non65_counts = df_non65['age_group'].value_counts().to_dict()
print("\nCurrent non-65 counts:")
print(current_non65_counts)

# Compute deficits (if target is higher than current) and surpluses (if target is lower than current)
deficits = {}
surpluses = {}
for group in raw_targets.keys():
    current = current_non65_counts.get(group, 0)
    target = target_counts[group]
    if current < target:
        deficits[group] = target - current
    elif current > target:
        surpluses[group] = current - target

print("\nDeficits (need additional rows):")
print(deficits)
print("\nSurpluses (rows to reassign):")
print(surpluses)

# Define a helper mapping for age ranges corresponding to each age group.
# Here we assume integer ages uniformly distributed in the range.
age_range_mapping = {
    "18-24": (18, 24),
    "25-33": (25, 33),
    "34-40": (34, 40),
    "41-50": (41, 50),
    "51-64": (51, 64)
}

# Function to randomly sample a new age from the given range (inclusive)
def sample_new_age(age_range):
    return np.random.randint(age_range[0], age_range[1] + 1)

# We will reassign rows from surplus groups to deficit groups.
# For each deficit group, we will pull rows from the available surplus bins.
# (A more sophisticated allocation might reassign proportionally;
# here we iterate over deficits and try to fill each deficit from available surplus.)

# Work on a copy of df_non65 to update rows
df_rebalanced = df_non65.copy()

# For each deficit group, fill the deficit by taking rows from surplus groups.
for target_group, deficit in deficits.items():
    print(f"\nFilling deficit for {target_group}: need {deficit} rows")
    remaining_deficit = deficit
    # Iterate over surplus groups (order can be chosen arbitrarily; here we loop over surpluses)
    for source_group in list(surpluses.keys()):
        available = surpluses[source_group]
        if available <= 0:
            continue
        # Determine how many rows to move from this surplus bin to the target
        num_to_move = min(available, remaining_deficit)
        print(f"  Reassigning {num_to_move} rows from {source_group} to {target_group}")
        # Identify candidate rows in source_group
        candidates = df_rebalanced[df_rebalanced['age_group'] == source_group]
        # Randomly sample indices to reassign
        indices_to_reassign = candidates.sample(n=num_to_move, random_state=42).index
        # Update these rows: assign a new 'approx_age' in the target group's range
        new_age_range = age_range_mapping[target_group]
        new_ages = [sample_new_age(new_age_range) for _ in range(num_to_move)]
        df_rebalanced.loc[indices_to_reassign, 'approx_age'] = new_ages
        # Also update the age_group column for these rows
        df_rebalanced.loc[indices_to_reassign, 'age_group'] = target_group
        
        # Update counters: decrease surplus for source_group and decrease remaining deficit
        surpluses[source_group] -= num_to_move
        remaining_deficit -= num_to_move
        if remaining_deficit <= 0:
            break
    if remaining_deficit > 0:
        print(f"Warning: Could not fill entire deficit for {target_group}. {remaining_deficit} rows remain unfilled.")

# Now, combine the rebalanced non-65 rows with the 65+ rows (which were not changed)
df_final = pd.concat([df_rebalanced, df[df['age_group'] == "65+"]])

# Recalculate final counts for non-65 groups
final_non65_counts = df_final[df_final['age_group'] != "65+"]['age_group'].value_counts().sort_index()
print("\nFinal non-65 counts after rebalancing:")
print(final_non65_counts)

# (Optional) Save the rebalanced dataset to a new CSV file
df_final.to_csv("rebalanced_age_data_two.csv", index=False)
print("\nRebalanced data saved to 'rebalanced_age_data.csv'.")

Current counts for each age group:
age_group
18-24    16262
25-33    91581
34-40    46788
41-50    23007
51-64     6045
65+        507
Name: count, dtype: int64

Target counts for non-65 age groups (normalized):
{'18-24': 45926, '25-33': 64297, '34-40': 45926, '41-50': 18370, '51-64': 9185}

Current non-65 counts:
{'25-33': 91581, '34-40': 46788, '41-50': 23007, '18-24': 16262, '51-64': 6045, '65+': 0}

Deficits (need additional rows):
{'18-24': 29664, '51-64': 3140}

Surpluses (rows to reassign):
{'25-33': 27284, '34-40': 862, '41-50': 4637}

Filling deficit for 18-24: need 29664 rows
  Reassigning 27284 rows from 25-33 to 18-24
  Reassigning 862 rows from 34-40 to 18-24
  Reassigning 1518 rows from 41-50 to 18-24

Filling deficit for 51-64: need 3140 rows
  Reassigning 3119 rows from 41-50 to 51-64

Final non-65 counts after rebalancing:
age_group
18-24    45926
25-33    64297
34-40    45926
41-50    18370
51-64     9164
65+          0
Name: count, dtype: int64

Rebalanced data saved